In [108]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [114]:
symbol = "SOLUSDT"
interval = "1d"
limit = 500

url = f"https://data-api.binance.vision/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
response = requests.get(url)
data = response.json()

df = pd.DataFrame(data, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume',
                                 'close_time', 'qav', 'num_trades', 'tbb', 'tbq', 'ignore'])
                                #fields were generated by ai using api documentation
df = df[['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
df['Date'] = pd.to_datetime(df['timestamp'], unit='ms')
df.set_index('Date', inplace=True)
print(df)

#Moving Average Crossover
short_ma = 30
long_ma = 80

df['Short_MA'] = df['Close'].rolling(window=short_ma).mean()
df['Long_MA'] = df['Close'].rolling(window=long_ma).mean()

df['Signal'] = 0
df['Signal'] = np.where(df['Short_MA'] > df['Long_MA'], 1, 0) #1 is to buy, 0 to stay still. Where method can be used like IF
df['Position'] = df['Signal'].diff() # after last activity calculate difference to understand weather we need to buy, sell or do nothing (1, -1, 0)

               timestamp    Open    High     Low   Close       Volume
Date                                                                 
2025-03-11  1.741651e+12  118.32  128.43  112.00  125.35  6435277.863
2025-03-12  1.741738e+12  125.35  131.33  121.22  126.62  4410502.140
2025-03-13  1.741824e+12  126.61  128.76  120.76  123.37  2824584.961
2025-03-14  1.741910e+12  123.36  136.03  122.98  133.54  3607725.492
2025-03-15  1.741997e+12  133.53  136.53  132.44  135.86  1918077.490
...                  ...     ...     ...     ...     ...          ...
2026-07-19  1.784419e+12   75.53   76.70   75.37   76.38  1176910.446
2026-07-20  1.784506e+12   76.38   78.38   75.50   77.85  1867200.851
2026-07-21  1.784592e+12   77.85   78.88   77.42   78.12  1283561.320
2026-07-22  1.784678e+12   78.13   78.85   77.00   77.97  1424652.173
2026-07-23  1.784765e+12   77.97   78.54   77.04   77.14   389075.003

[500 rows x 6 columns]


In [120]:
#-------–––––-------------PL------------------------------
market_returns = [0.0]
historic_based = [0.0]
cumulated = [1.0] # initial capital

# convert df to list to make it easier
closes = list(df['Close'])
signals = list(df['Signal'])

for i in range(1, len(closes)): # start from the next day to get te previous where index 0
    returns = (closes[i] - closes[i-1]) / closes[i-1] # (Ціна сьогодні - Ціна вчора )/ Ціна_вчора
    market_returns.append(returns) #what we've basically calculated is the change in crypto price itself

    # using historic data we look at the previous day we can calculate how much in % did we gain (since signal can be either 1 or 0), which mean that when we just wait, our revenue is 0, but then sell, we gain the price of the unit
    unit_historical = returns * signals[i-1]
    historic_based.append(unit_historical)

    #накопичений результат за вчора на (1 + дохід за сьогодні)
    cum_strat = cumulated[i-1] * (1 + unit_historical)
    cumulated.append(cum_strat)

# rewrite back to df
df['market_returns'] = market_returns
df['historic_based'] = historic_based
df['cumulated'] = cumulated


final_strategy_balance = df['cumulated'].iloc[-1]
final_market_balance = (1 + df['market_returns']).cumprod().iloc[-1]
print(f"Initial capital: 1.00 (100%)")
print(f"Final balance of P&L: {final_strategy_balance:.2f}")

Initial capital: 1.00 (100%)
Final balance of P&L: 0.50


In [119]:
days_to_predict = 30
prices = df['Close'].values
n = len(prices)
# fourier transform: look for hidden waves
fft_vals = np.fft.fft(prices)
fft_frequents = np.fft.fftfreq(n)

# Для реалізації рядів був залучений Джеміні, задано питання про те, за якої логікою накладаються синусоїдні хвилі та які методи бібліотек в цьому допоможуть.

# Залишаємо лише 15 найсильніших хвиль (щоб прибрати ринковий шум)
top_harmonics = 15
top_indices = np.argsort(np.abs(fft_vals))[::-1][:top_harmonics]

# Створюємо масив часу: від минулого до майбутнього (500 + 30 днів)
in_limit = np.arange(n)
plus_prediction = np.arange(n + days_to_predict)
forecast = np.zeros(len(plus_prediction), dtype=complex)

for i in top_indices:
    amplitude = fft_vals[i] / n
    frequency = fft_frequents[i]
    forecast += amplitude * np.exp(
        1j * 2 * np.pi * frequency * plus_prediction)  #за формулою Ейлера 1j потрібна нам для того щоб np.exp перетворила функцію на синусоїду. в наступному рядку ми працюватимемо в полі дійсних чисел, тому комплексна зникне
forecast = np.real(forecast)

# Прогенеровуємо дати з останньої до тої, яку ми прогнозуємо. Був використаний Джеміні, щоб зрозуміти, які методи для цього потрібні.
last_date = df.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=days_to_predict)